# Dataset Validation and Integrity Check

Validate the ImageNet100 dataset for potential issues:
- Missing or corrupt images
- Class imbalance
- Train/Val consistency
- File integrity

In [8]:
from pathlib import Path
from PIL import Image
import json
from collections import defaultdict
import pandas as pd

# Paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'imagenet100'
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'val'
LABELS_JSON = DATA_ROOT / 'Labels.json'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Train dir exists: {TRAIN_DIR.exists()}")
print(f"Val dir exists: {VAL_DIR.exists()}")

Notebook dir: c:\Users\giova\Universita\deep_learning\Masked-Autoencoders-for-Image-Representation-Learning\notebooks
Project root: c:\Users\giova\Universita\deep_learning\Masked-Autoencoders-for-Image-Representation-Learning
Data root: c:\Users\giova\Universita\deep_learning\Masked-Autoencoders-for-Image-Representation-Learning\data\raw\imagenet100
Train dir exists: True
Val dir exists: True


## Function to validate split

In [9]:
def validate_split(split_dir, split_name='train', max_errors=10):
    """
    Validate a split directory (train or val).
    Returns: dict with validation results
    """
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPEG'}
    
    results = {
        'split': split_name,
        'total_classes': 0,
        'total_images': 0,
        'corrupt_images': [],
        'missing_files': [],
        'class_counts': {},
        'image_sizes': defaultdict(list),
        'errors': []
    }
    
    if not split_dir.exists():
        results['errors'].append(f"Split directory not found: {split_dir}")
        return results
    
    # Iterate over classes
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        
        class_id = class_dir.name
        results['total_classes'] += 1
        image_count = 0
        
        # Iterate over images in class
        for img_file in class_dir.iterdir():
            if img_file.suffix.lower() in image_extensions:
                try:
                    # Try to open and validate image
                    img = Image.open(img_file)
                    img.load()  # Force load to detect corruption
                    results['image_sizes'][class_id].append(img.size)
                    image_count += 1
                    results['total_images'] += 1
                except Exception as e:
                    if len(results['corrupt_images']) < max_errors:
                        results['corrupt_images'].append((str(img_file), str(e)))
                    else:
                        results['errors'].append(f"... and more corrupt images")
                        break
        
        results['class_counts'][class_id] = image_count
    
    return results

print("Validation functions loaded")

Validation functions loaded


## Validate train and val splits

In [10]:
print("Validating train split...")
train_results = validate_split(TRAIN_DIR, split_name='train')

print("\nTrain Split Results:")
print(f"  Total classes: {train_results['total_classes']}")
print(f"  Total images: {train_results['total_images']}")
print(f"  Corrupt images: {len(train_results['corrupt_images'])}")
if train_results['corrupt_images']:
    print("  Corrupt image samples:")
    for img_path, error in train_results['corrupt_images'][:3]:
        print(f"    - {img_path}: {error}")
if train_results['errors']:
    print(f"  Errors: {', '.join(train_results['errors'])}")

print("\nValidating val split...")
val_results = validate_split(VAL_DIR, split_name='val')

print("\nVal Split Results:")
print(f"  Total classes: {val_results['total_classes']}")
print(f"  Total images: {val_results['total_images']}")
print(f"  Corrupt images: {len(val_results['corrupt_images'])}")
if val_results['corrupt_images']:
    print("  Corrupt image samples:")
    for img_path, error in val_results['corrupt_images'][:3]:
        print(f"    - {img_path}: {error}")
if val_results['errors']:
    print(f"  Errors: {', '.join(val_results['errors'])}")

Validating train split...


KeyboardInterrupt: 

## Check train/val consistency

In [5]:
train_classes = set(train_results['class_counts'].keys())
val_classes = set(val_results['class_counts'].keys())

missing_in_val = sorted(train_classes - val_classes)
missing_in_train = sorted(val_classes - train_classes)
common_classes = sorted(train_classes & val_classes)

print("\nTrain/Val Consistency:")
print(f"  Classes in train: {len(train_classes)}")
print(f"  Classes in val: {len(val_classes)}")
print(f"  Common classes: {len(common_classes)}")
print(f"  Missing in val: {len(missing_in_val)}")
print(f"  Missing in train: {len(missing_in_train)}")

if missing_in_val:
    print(f"\n  Classes in train but not val: {missing_in_val[:5]} {'...' if len(missing_in_val) > 5 else ''}")
if missing_in_train:
    print(f"  Classes in val but not train: {missing_in_train[:5]} {'...' if len(missing_in_train) > 5 else ''}")


Train/Val Consistency:
  Classes in train: 0
  Classes in val: 0
  Common classes: 0
  Missing in val: 0
  Missing in train: 0


## Class distribution analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# For common classes, compare train vs val counts
train_counts_common = [train_results['class_counts'][c] for c in common_classes]
val_counts_common = [val_results['class_counts'][c] for c in common_classes]

print(f"\nClass Distribution (common classes):")
print(f"  Train - Mean: {np.mean(train_counts_common):.1f}, Std: {np.std(train_counts_common):.1f}")
print(f"  Train - Min: {min(train_counts_common)}, Max: {max(train_counts_common)}")
print(f"  Val - Mean: {np.mean(val_counts_common):.1f}, Std: {np.std(val_counts_common):.1f}")
print(f"  Val - Min: {min(val_counts_common)}, Max: {max(val_counts_common)}")

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_counts_common, bins=30, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Images per class')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Train Distribution (common classes)')
axes[0].grid(alpha=0.3)
axes[0].axvline(np.mean(train_counts_common), color='red', linestyle='--', label=f'Mean: {np.mean(train_counts_common):.0f}')
axes[0].legend()

axes[1].hist(val_counts_common, bins=30, alpha=0.7, edgecolor='black', color='orange')
axes[1].set_xlabel('Images per class')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Val Distribution (common classes)')
axes[1].grid(alpha=0.3)
axes[1].axvline(np.mean(val_counts_common), color='red', linestyle='--', label=f'Mean: {np.mean(val_counts_common):.0f}')
axes[1].legend()

plt.tight_layout()
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIGURES_DIR / '03_class_distribution_validation.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Saved class distribution plot to {FIGURES_DIR / '03_class_distribution_validation.png'}")


Class Distribution (common classes):
  Train - Mean: nan, Std: nan


c:\Users\giova\Universita\deep_learning\Masked-Autoencoders-for-Image-Representation-Learning\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\giova\Universita\deep_learning\Masked-Autoencoders-for-Image-Representation-Learning\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\giova\Universita\deep_learning\Masked-Autoencoders-for-Image-Representation-Learning\.venv\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\giova\Universita\deep_learning\Masked-Autoencoders-for-Image-Representation-Learning\.venv\Lib\site-packages\numpy\_core\_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\

ValueError: min() arg is an empty sequence

## Check image sizes per class

In [ ]:
# Sample image sizes
all_sizes = []
for sizes in train_results['image_sizes'].values():
    all_sizes.extend(sizes)

if all_sizes:
    widths = [s[0] for s in all_sizes]
    heights = [s[1] for s in all_sizes]
    
    print(f"\nImage Size Statistics (train):")
    print(f"  Width - Mean: {np.mean(widths):.1f}, Min: {min(widths)}, Max: {max(widths)}")
    print(f"  Height - Mean: {np.mean(heights):.1f}, Min: {min(heights)}, Max: {max(heights)}")
    
    # Check aspect ratio
    aspect_ratios = [w/h for w, h in all_sizes]
    print(f"  Aspect Ratio - Mean: {np.mean(aspect_ratios):.2f}, Min: {min(aspect_ratios):.2f}, Max: {max(aspect_ratios):.2f}")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(widths, heights, alpha=0.3, s=20)
    axes[0].set_xlabel('Width')
    axes[0].set_ylabel('Height')
    axes[0].set_title(f'Image Sizes (n={len(all_sizes)})')
    axes[0].grid(alpha=0.3)
    
    axes[1].hist(aspect_ratios, bins=30, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Aspect Ratio (W/H)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Aspect Ratio Distribution')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR.parent.parent / 'figures' / '03_image_sizes_validation.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Saved image size plots")
else:
    print("No images found - cannot compute size statistics")

## Save validation report

In [ ]:
# Create summary report
report = {
    'dataset': 'ImageNet100',
    'validation_date': pd.Timestamp.now().isoformat(),
    'train': {
        'classes': train_results['total_classes'],
        'images': train_results['total_images'],
        'corrupt': len(train_results['corrupt_images']),
        'images_per_class': {
            'mean': float(np.mean(train_counts_common)) if train_counts_common else 0,
            'std': float(np.std(train_counts_common)) if train_counts_common else 0,
            'min': int(min(train_counts_common)) if train_counts_common else 0,
            'max': int(max(train_counts_common)) if train_counts_common else 0
        }
    },
    'val': {
        'classes': val_results['total_classes'],
        'images': val_results['total_images'],
        'corrupt': len(val_results['corrupt_images']),
        'images_per_class': {
            'mean': float(np.mean(val_counts_common)) if val_counts_common else 0,
            'std': float(np.std(val_counts_common)) if val_counts_common else 0,
            'min': int(min(val_counts_common)) if val_counts_common else 0,
            'max': int(max(val_counts_common)) if val_counts_common else 0
        }
    },
    'consistency': {
        'common_classes': len(common_classes),
        'missing_in_val': len(missing_in_val),
        'missing_in_train': len(missing_in_train)
    },
    'image_sizes': {
        'width': {'mean': float(np.mean(widths)) if all_sizes else 0, 'min': int(min(widths)) if all_sizes else 0, 'max': int(max(widths)) if all_sizes else 0},
        'height': {'mean': float(np.mean(heights)) if all_sizes else 0, 'min': int(min(heights)) if all_sizes else 0, 'max': int(max(heights)) if all_sizes else 0},
        'aspect_ratio': {'mean': float(np.mean(aspect_ratios)) if all_sizes else 0, 'min': float(min(aspect_ratios)) if all_sizes else 0, 'max': float(max(aspect_ratios)) if all_sizes else 0}
    }
}

# Save report
report_path = PROCESSED_DIR / 'imagenet100_validation_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nValidation report saved to {report_path}")
print(json.dumps(report, indent=2))

## Summary

In [ ]:
print("\n" + "="*60)
print("VALIDATION SUMMARY")
print("="*60)
print(f"\nStatus: {'✓ PASSED' if len(train_results['corrupt_images']) == 0 and len(missing_in_val) == 0 else '✗ ISSUES FOUND'}")
print(f"\nDataset Total:")
print(f"  Total classes (common): {len(common_classes)}")
print(f"  Total train images: {train_results['total_images']}")
print(f"  Total val images: {val_results['total_images']}")
print(f"  Grand total: {train_results['total_images'] + val_results['total_images']} images")
print(f"\nCorruption Status:")
print(f"  Corrupt train images: {len(train_results['corrupt_images'])}")
print(f"  Corrupt val images: {len(val_results['corrupt_images'])}")
print(f"\nClass Consistency:")
print(f"  Classes in train only: {len(missing_in_val)}")
print(f"  Classes in val only: {len(missing_in_train)}")
print(f"\nRecommendations:")
if len(train_results['corrupt_images']) > 0 or len(val_results['corrupt_images']) > 0:
    print("  ⚠ Remove or redownload corrupt images")
if len(missing_in_val) > 0 or len(missing_in_train) > 0:
    print("  ⚠ Ensure train and val have same classes")
if len(train_results['corrupt_images']) == 0 and len(val_results['corrupt_images']) == 0 and len(missing_in_val) == 0 and len(missing_in_train) == 0:
    print("  ✓ Dataset is ready for training")